# SSPNet 작은 수영자 재현율 우선 추가 학습

프로젝트에서 검토·승인한 `recall_v1.zip`을 사용합니다. YoloBeachTrain 전체 ZIP은 사용하지 않습니다.
원본(epoch 0)과 매 epoch를 검증합니다. 이미지당 FP 예산 안에서 원본 좌표 √면적 2~20픽셀 수영자의 재현율을 최우선으로 선택합니다.
동점이면 전체 재현율, 낮은 FP 순서입니다. 학습이 원본보다 나쁘면 원본을 유지합니다.
현재 데이터는 소규모 초기 실험 묶음입니다. 일반화 성능을 보장하지 않으며 추가 이미지는 검토 후 별도 버전으로 준비합니다.
무료 GPU를 선택하고 순서대로 실행하세요. Drive 인증은 직접 완료합니다.


In [ ]:
import os,sys,json,subprocess,zipfile
from pathlib import Path
from google.colab import drive,files
drive.mount('/content/drive')
RUN_NAME='recall_v1_3epochs'
EPOCHS=3
LEARNING_RATE=0.00001
FP_PER_IMAGE=4.0
DATA_ZIP_PATH='' # Drive의 recall_v1.zip 경로. 비우면 업로드합니다.
WORK=Path('/content/sspnet_free');WORK.mkdir(exist_ok=True)
STORE=Path('/content/drive/MyDrive/SSPNet_free_colab');STORE.mkdir(parents=True,exist_ok=True)
if not RUN_NAME or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_NAME):
    raise ValueError('실행 이름은 영문·숫자·밑줄·하이픈으로 지정하세요.')


In [ ]:
HELPER_SOURCES = {'setup_environment.py': '"""무료 Colab의 기본 Python과 분리하여 SSPNet 실행 환경을 만든다."""\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nROOT = Path("/content/sspnet_free")\nREPO = ROOT / "SSPNet"\nVENV = ROOT / "venv"\nPYTHON = VENV / "bin/python"\nCOMMIT = "f2d0f42273945577a5c6497fd2b384cb545f8f66"\nSTAMP = "python38-torch110-cu111-mmcv1318-v1"\n\n\ndef command(*args, env=None):\n    subprocess.run([str(a) for a in args], check=True, env=env)\n\n\ndef setup():\n    if platform.system() != "Linux" or platform.machine() != "x86_64" or not Path("/content").is_dir():\n        raise RuntimeError("이 설치 셀은 Google Colab Linux 런타임에서 실행하세요.")\n    if not shutil.which("nvidia-smi"):\n        raise RuntimeError("무료 Colab에서 런타임 유형 변경 → GPU를 선택한 뒤 다시 실행하세요.")\n    command("nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv")\n    free = shutil.disk_usage("/content").free / 1024**3\n    print("Colab 임시 디스크 여유: %.1f GiB" % free, flush=True)\n    if free < 8:\n        raise RuntimeError("환경 설치를 시작하려면 임시 디스크에 최소 8 GiB 여유가 필요합니다.")\n    ROOT.mkdir(exist_ok=True)\n    if not PYTHON.is_file():\n        command(sys.executable,"-m","pip","install","--quiet","--no-cache-dir","uv==0.9.7")\n        command("uv","python","install","3.8.20")\n        command("uv","venv","--python","3.8.20","--seed",VENV)\n    env = os.environ.copy()\n    env.pop("PYTHONPATH",None)\n    env.pop("PYTHONHOME",None)\n    env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"\n    env["PYTHONNOUSERSITE"] = "1"\n    marker = ROOT / "environment_ready.json"\n    ready = json.loads(marker.read_text()).get("stamp") == STAMP if marker.exists() else False\n    if not ready:\n        command(PYTHON,"-m","pip","install","--no-cache-dir","pip==24.3.1",\n                "setuptools==59.5.0","wheel==0.45.1",env=env)\n        command(PYTHON,"-m","pip","install","--no-cache-dir",\n                "torch==1.10.0+cu111","torchvision==0.11.1+cu111",\n                "--find-links","https://download.pytorch.org/whl/cu111/torch_stable.html",env=env)\n        command(PYTHON,"-m","pip","install","--no-cache-dir",\n                "numpy==1.21.6","scipy==1.7.3","matplotlib==3.5.3","Pillow==9.5.0",\n                "opencv-python-headless==4.5.5.64","pycocotools==2.0.7",\n                "addict==2.4.0","yapf==0.31.0","PyYAML==6.0.1","packaging==23.2",\n                "terminaltables==3.1.10","six==1.16.0","tqdm==4.66.5","gdown==5.2.0",env=env)\n        # 사전 빌드된 연산자만 설치한다. 무료 GPU 시간으로 CUDA 소스 빌드를 하지 않는다.\n        command(PYTHON,"-m","pip","install","--no-cache-dir","--no-deps",\n                "https://download.openmmlab.com/mmcv/dist/cu111/torch1.10.0/"\n                "mmcv_full-1.3.18-cp38-cp38-manylinux1_x86_64.whl",env=env)\n    if not REPO.exists():\n        command("git","init",REPO)\n        command("git","-C",REPO,"remote","add","origin","https://github.com/MingboHong/SSPNet.git")\n        command("git","-C",REPO,"fetch","--depth=1","origin",COMMIT)\n        command("git","-C",REPO,"checkout","--detach","FETCH_HEAD")\n    actual = subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()\n    if actual != COMMIT:\n        raise RuntimeError("기존 SSPNet 코드 버전이 다릅니다. 새 무료 런타임에서 설치 셀부터 실행하세요.")\n    env["PYTHONPATH"] = str(REPO)\n    check = \'\'\'\nimport json, sys, torch, torchvision, mmcv, mmdet\nfrom mmcv.ops import nms, roi_align\nassert torch.__version__ == \'1.10.0+cu111\'\nassert mmcv.__version__ == \'1.3.18\'\nassert torch.cuda.is_available(), \'GPU를 사용할 수 없습니다.\'\na = torch.randn(32,32,device=\'cuda\'); (a @ a).sum().item()\nb = torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]],device=\'cuda\')\ns = torch.tensor([.9,.8],device=\'cuda\')\nnms(b,s,.5)\nroi_align(torch.ones(1,1,16,16,device=\'cuda\'),torch.tensor([[0.,1.,1.,8.,8.]],device=\'cuda\'),(3,3))\ntorch.cuda.synchronize()\nprint(json.dumps(dict(python=sys.version,torch=torch.__version__,torchvision=torchvision.__version__,\n    cuda=torch.version.cuda,mmcv=mmcv.__version__,mmdet=mmdet.__version__,\n    gpu=torch.cuda.get_device_name(0),capability=torch.cuda.get_device_capability(0)),ensure_ascii=False,indent=2))\n\'\'\'\n    command(PYTHON,"-c",check,env=env)\n    marker.write_text(json.dumps(dict(stamp=STAMP,source_commit=actual)),encoding="utf-8")\n    freeze = subprocess.check_output([str(PYTHON),"-m","pip","freeze"],env=env,text=True)\n    (ROOT/"installed_packages.txt").write_text(freeze,encoding="utf-8")\n    print("환경 확인 완료: CUDA 행렬 연산, NMS, RoIAlign, SSPNet 패키지 가져오기 성공",flush=True)\n\n\nif __name__ == "__main__":\n    setup()\n', 'metrics.py': '"""원본 이미지 좌표에서 수영자 탐지와 크기별 재현율을 평가한다."""\nimport json\nimport math\nimport re\nimport os\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\n\ndef canonical(value):\n    return " ".join(re.sub(r"[_-]+", " ", value.lower()).split())\n\n\ndef image_path(annotation_path, name):\n    base = Path(annotation_path).resolve().parent\n    path = (base / name).resolve()\n    if base != path and base not in path.parents:\n        raise ValueError("이미지 경로가 데이터셋 폴더 밖을 가리킵니다: " + name)\n    if not path.is_file():\n        raise FileNotFoundError("정답 파일과 같은 폴더 아래에 이미지가 필요합니다: " + str(path))\n    return path\n\n\ndef size_group(box):\n    d = max(float(box[2]), float(box[3]))\n    return "0<d<=8" if d <= 8 else "8<d<=12" if d <= 12 else "12<d<=20" if d <= 20 else "20<d"\n\n\ndef geometric_size(box):\n    return math.sqrt(float(box[2]) * float(box[3]))\n\n\ndef merge_splits(annotation_paths, output_path):\n    """분할별로 겹치는 ID와 파일명을 구분하고 이미지 복사 없이 합친다."""\n    output_path = Path(output_path).resolve()\n    base = output_path.parent\n    merged = dict(info={"description": "전체 분할 통합", "source_splits": []},\n                  licenses=[], images=[], annotations=[], categories=[])\n    categories = {}\n    seen_paths = set()\n    for split, source in annotation_paths:\n        source = Path(source).resolve()\n        if split in merged["info"]["source_splits"]:\n            raise ValueError("같은 분할을 두 번 합칠 수 없습니다: " + split)\n        raw = json.loads(source.read_text(encoding="utf-8"))\n        merged["info"]["source_splits"].append(split)\n        catmap = {}\n        for category in raw["categories"]:\n            name = canonical(category["name"])\n            if name not in categories:\n                categories[name] = len(categories) + 1\n                merged["categories"].append(dict(id=categories[name], name=category["name"]))\n            catmap[category["id"]] = categories[name]\n        images = {}\n        for im in sorted(raw["images"], key=lambda v: v["id"]):\n            if im["id"] in images:\n                raise ValueError("분할 안에서 이미지 ID가 중복됩니다: " + split)\n            path = image_path(source, im["file_name"])\n            if path in seen_paths:\n                raise ValueError("같은 이미지 경로가 중복됩니다: " + str(path))\n            seen_paths.add(path)\n            if base not in path.parents:\n                raise ValueError("통합 정답은 모든 이미지의 상위 폴더에 저장해야 합니다.")\n            new_id = len(merged["images"])\n            images[im["id"]] = new_id\n            merged["images"].append(dict(im, id=new_id, file_name=path.relative_to(base).as_posix(),\n                                         source_split=split, source_image_id=im["id"]))\n        seen_annotations = set()\n        for a in raw["annotations"]:\n            if a["id"] in seen_annotations:\n                raise ValueError("분할 안에서 정답 ID가 중복됩니다: " + split)\n            seen_annotations.add(a["id"])\n            if a["image_id"] not in images or a["category_id"] not in catmap:\n                raise ValueError("정답의 이미지 또는 종류 ID를 찾지 못했습니다: " + split)\n            merged["annotations"].append(dict(a, id=len(merged["annotations"])+1,\n                image_id=images[a["image_id"]], category_id=catmap[a["category_id"]],\n                source_split=split, source_annotation_id=a["id"]))\n    if not merged["images"]:\n        raise ValueError("합칠 이미지가 없습니다.")\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = output_path.with_suffix(".json.tmp")\n    temporary.write_text(json.dumps(merged, ensure_ascii=False, indent=2), encoding="utf-8")\n    os.replace(str(temporary), str(output_path))\n    return output_path\n\n\ndef load_dataset(annotation_path, target_names=("person water",), limit=0):\n    raw = json.loads(Path(annotation_path).read_text(encoding="utf-8"))\n    cats = raw.get("categories", [])\n    wanted = {canonical(s) for s in target_names}\n    selected = [c["id"] for c in cats if canonical(c["name"]) in wanted]\n    if not selected:\n        raise ValueError("물속 사람 라벨을 찾지 못했습니다. 실제 종류: " + str([c["name"] for c in cats]))\n    images = sorted(raw["images"], key=lambda im: im["id"])\n    if len({im["id"] for im in images}) != len(images):\n        raise ValueError("이미지 ID가 중복되어 있습니다.")\n    if limit < 0:\n        raise ValueError("이미지 제한은 0 이상이어야 합니다.")\n    if limit:\n        images = images[:limit]\n    if not images:\n        raise ValueError("선택한 분할에 이미지가 없습니다.")\n    imap = {im["id"]: im for im in images}\n    ann_ids = set()\n    annotations = []\n    for a in raw["annotations"]:\n        if a["id"] in ann_ids:\n            raise ValueError("정답 ID가 중복되어 있습니다.")\n        ann_ids.add(a["id"])\n        if a["image_id"] not in imap or a["category_id"] not in selected:\n            continue\n        x, y, w, h = map(float, a["bbox"])\n        im = imap[a["image_id"]]\n        if not all(math.isfinite(v) for v in (x, y, w, h)) or w <= 0 or h <= 0:\n            raise ValueError("유효하지 않은 정답 박스: " + str(a["id"]))\n        if x < -.5 or y < -.5 or x + w > im["width"] + .5 or y + h > im["height"] + .5:\n            raise ValueError("이미지 밖의 정답 박스입니다. 좌표 변환을 확인하세요: " + str(a["id"]))\n        annotations.append(dict(id=a["id"], image_id=a["image_id"], category_id=1,\n                                bbox=[x, y, w, h], area=w*h,\n                                iscrowd=int(a.get("iscrowd", 0)),\n                                ignore=bool(a.get("ignore", False)), segmentation=[]))\n    for im in images:\n        path = image_path(annotation_path, im["file_name"])\n        from PIL import Image\n        with Image.open(path) as pic:\n            if pic.size != (im["width"], im["height"]):\n                raise ValueError("이미지 크기와 정답 메타데이터가 다릅니다: " + str(path))\n    gt = dict(info={"description": "물속 사람 평가용 정답"}, licenses=[], images=images,\n              annotations=annotations, categories=[dict(id=1, name="person water")])\n    counts = Counter(size_group(a["bbox"]) for a in annotations if not a["ignore"] and not a["iscrowd"])\n    normal = [a for a in annotations if not a["ignore"] and not a["iscrowd"]]\n    split_images = Counter(im.get("source_split", "single") for im in images)\n    image_splits = {im["id"]: im.get("source_split", "single") for im in images}\n    split_targets = Counter(image_splits[a["image_id"]] for a in normal)\n    return gt, dict(image_count=len(images), available_image_count=len(raw["images"]), target_count=sum(counts.values()),\n                    geometric_2to20_count=sum(2 <= geometric_size(a["bbox"]) < 20 for a in normal),\n                    geometric_2to8_count=sum(2 <= geometric_size(a["bbox"]) < 8 for a in normal),\n                    split_counts={s: dict(images=n, water_targets=split_targets[s]) for s,n in split_images.items()},\n                    size_counts=dict(counts), available_classes=[c["name"] for c in cats],\n                    target_classes=[c["name"] for c in cats if c["id"] in selected])\n\n\ndef overlap(a, b, crowd=False):\n    x, y, w, h = a\n    u, v, s, t = b\n    inter = max(0., min(x+w, u+s)-max(x, u)) * max(0., min(y+h, v+t)-max(y, v))\n    area = w*h if crowd else w*h+s*t-inter\n    return inter / area if area > 0 else 0.\n\n\ndef match_predictions(gt, predictions, iou=.5):\n    normal, ignored = defaultdict(list), defaultdict(list)\n    for a in gt["annotations"]:\n        (ignored if a.get("ignore") or a.get("iscrowd") else normal)[a["image_id"]].append(a)\n    used = set()\n    valid_ids = {im["id"] for im in gt["images"]}\n    ranked = sorted((p for p in predictions if p["image_id"] in valid_ids), key=lambda p: -p["score"])\n    matches = []\n    for p in ranked:\n        candidates = [(overlap(p["bbox"], a["bbox"]), a) for a in normal[p["image_id"]] if a["id"] not in used]\n        score, best = max(candidates, key=lambda pair: pair[0]) if candidates else (0., None)\n        if best is not None and score >= iou:\n            used.add(best["id"])\n            matches.append((p, best, False))\n        elif any(overlap(p["bbox"], a["bbox"], bool(a.get("iscrowd"))) >= iou for a in ignored[p["image_id"]]):\n            matches.append((p, None, True))\n        else:\n            matches.append((p, None, False))\n    return matches\n\n\ndef evaluate(gt, predictions, score_threshold=.25, iou=.5):\n    matches = match_predictions(gt, predictions, iou)\n    counts = Counter(size_group(a["bbox"]) for a in gt["annotations"] if not a.get("ignore") and not a.get("iscrowd"))\n    positives = sum(counts.values())\n    hits, tp, fp = Counter(), 0, 0\n    recalls, precisions = [], []\n    total_tp = total_fp = 0\n    for p, a, ignored in matches:\n        if ignored:\n            continue\n        total_tp += a is not None\n        total_fp += a is None\n        recalls.append(total_tp / positives if positives else 0.)\n        precisions.append(total_tp / (total_tp + total_fp))\n        if p["score"] >= score_threshold:\n            tp += a is not None\n            fp += a is None\n            if a is not None:\n                hits[size_group(a["bbox"])] += 1\n    ap = sum(max((p for r, p in zip(recalls, precisions) if r >= q/100), default=0.) for q in range(101))/101 if positives else None\n    tiny_count = sum(counts[k] for k in ("0<d<=8", "8<d<=12", "12<d<=20"))\n    tiny_hits = sum(hits[k] for k in ("0<d<=8", "8<d<=12", "12<d<=20"))\n    def geometric_range(low, high):\n        selected = {a["id"] for a in gt["annotations"] if not a.get("ignore") and not a.get("iscrowd")\n                    and low <= geometric_size(a["bbox"]) < high}\n        found = sum(a is not None and a["id"] in selected for p,a,ignored in matches\n                    if not ignored and p["score"] >= score_threshold)\n        return dict(positives=len(selected), hits=found, recall=found/len(selected) if selected else None)\n    return dict(iou=iou, score_threshold=score_threshold, images=len(gt["images"]),\n                positives=positives, true_positives=tp, false_positives=fp,\n                precision=tp/(tp+fp) if tp+fp else None,\n                recall=tp/positives if positives else None,\n                false_positives_per_image=fp/len(gt["images"]),\n                AP50_saved_predictions_101point=ap if iou == .5 else None,\n                geometric_size_definition="s=sqrt(width*height), 배포 이미지 픽셀 기준",\n                geometric_2to20=geometric_range(2,20),\n                geometric_2to8=geometric_range(2,8),\n                geometric_size_recall={"2<=s<8": geometric_range(2,8),\n                                       "8<=s<12": geometric_range(8,12),\n                                       "12<=s<20": geometric_range(12,20)},\n                tiny_le20=dict(positives=tiny_count, hits=tiny_hits,\n                               recall=tiny_hits/tiny_count if tiny_count else None),\n                size_recall={k: dict(positives=counts[k], hits=hits[k],\n                                    recall=hits[k]/counts[k] if counts[k] else None)\n                             for k in ("0<d<=8", "8<d<=12", "12<d<=20", "20<d")})\n', 'recall_selection.py': '"""작은 수영자 재현율과 오탐 예산으로 검증 모델을 선택한다. GPU 의존성 없음."""\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nfrom metrics import match_predictions\n\n\ndef digest(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\n\n\ndef verify_bundle(root):\n    root=Path(root).resolve()\n    manifest=json.loads((root/\'quality_manifest.json\').read_text(encoding=\'utf-8\'))\n    groups={};hashes={};data={}\n    for split in (\'train\',\'val\'):\n        entry=manifest[\'splits\'][split]\n        ann=(root/entry[\'annotation_file\']).resolve()\n        if root not in ann.parents or digest(ann)!=entry[\'annotation_sha256\']:\n            raise ValueError(\'승인 후 주석이 변경됐거나 데이터 경로가 올바르지 않습니다.\')\n        raw=json.loads(ann.read_text(encoding=\'utf-8\'))\n        images={im[\'id\']:im for im in raw[\'images\']}\n        if len(images)!=len(raw[\'images\']) or {r[\'image_id\'] for r in entry[\'images\']}!=set(images):\n            raise ValueError(\'검토 목록과 이미지 목록이 다릅니다.\')\n        hashes[split]=set();groups[split]=set()\n        for item in entry[\'images\']:\n            if item.get(\'review\',{}).get(\'status\')!=\'accepted\':raise ValueError(\'주석 검토 승인 없는 이미지입니다.\')\n            path=(root/item[\'file_name\']).resolve()\n            if root not in path.parents or item[\'file_name\']!=images[item[\'image_id\']][\'file_name\']:\n                raise ValueError(\'이미지 경로가 데이터 범위를 벗어납니다.\')\n            if digest(path)!=item[\'sha256\']:raise ValueError(\'승인 후 이미지가 변경됐습니다.\')\n            if item[\'group\']!=images[item[\'image_id\']][\'capture_group\']:raise ValueError(\'촬영 그룹이 일치하지 않습니다.\')\n            groups[split].add(item[\'group\']);hashes[split].add(item[\'sha256\'])\n        if groups[split]!=set(entry[\'groups\']):raise ValueError(\'촬영 그룹 목록이 다릅니다.\')\n        ids=set()\n        for a in raw[\'annotations\']:\n            if a[\'id\'] in ids or a[\'image_id\'] not in images:raise ValueError(\'중복 또는 연결되지 않은 주석입니다.\')\n            ids.add(a[\'id\'])\n            im=images[a[\'image_id\']];x,y,w,h=a[\'bbox\']\n            if not all(math.isfinite(v) for v in (x,y,w,h)) or min(w,h)<1 or min(x,y)<0 or x+w>im[\'width\'] or y+h>im[\'height\']:\n                raise ValueError(\'유효하지 않은 사람 박스입니다.\')\n            if a[\'category_id\']!=1 or not isinstance(a.get(\'water\'),bool):raise ValueError(\'사람 클래스와 물속 여부 주석을 확인하세요.\')\n        data[split]=raw\n    if groups[\'train\']&groups[\'val\'] or hashes[\'train\']&hashes[\'val\']:\n        raise ValueError(\'학습과 검증에 같은 촬영 또는 이미지가 있습니다.\')\n    if (groups[\'train\']|groups[\'val\'])&set(manifest[\'reserved_groups\']):\n        raise ValueError(\'보존 평가 촬영이 학습·검증에 포함됐습니다.\')\n    return manifest,data\n\n\ndef select_operating_point(gt,predictions,fp_per_image=4.,iou=.3,small_low=2.,small_high=20.,\n                           thresholds=(.001,.005,.01,.025,.05,.075,.1,.15,.2,.25,.35,.5,.75)):\n    if fp_per_image<0 or not 0<iou<=1 or not 0<=small_low<small_high:raise ValueError(\'검증 기준이 올바르지 않습니다.\')\n    if not gt[\'images\']:raise ValueError(\'검증 이미지가 없습니다.\')\n    for prediction in predictions:\n        if not math.isfinite(prediction[\'score\']) or not 0<=prediction[\'score\']<=1 or not all(math.isfinite(v) for v in prediction[\'bbox\']):\n            raise ValueError(\'유효하지 않은 검증 예측입니다.\')\n    normal=[a for a in gt[\'annotations\'] if not a.get(\'ignore\') and not a.get(\'iscrowd\')]\n    small={a[\'id\'] for a in normal if a.get(\'water\') and small_low<=math.sqrt(a[\'bbox\'][2]*a[\'bbox\'][3])<small_high}\n    if not small:raise ValueError(\'검증 데이터에 작은 수영자 정답이 없습니다.\')\n    matches=match_predictions(gt,predictions,iou)\n    points=[]\n    for threshold in tuple(thresholds)+(1.000001,):\n        selected=[(p,a) for p,a,ignored in matches if not ignored and p[\'score\']>=threshold]\n        hit={a[\'id\'] for p,a in selected if a is not None}\n        fp=sum(a is None for p,a in selected)\n        points.append(dict(threshold=threshold,small_swimmer_positives=len(small),small_swimmer_hits=len(small&hit),\n                           small_swimmer_recall=len(small&hit)/len(small),person_recall=len(hit)/len(normal) if normal else 0.,\n                           TP=len(hit),FP=fp,FN=len(normal)-len(hit),FP_per_image=fp/len(gt[\'images\'])))\n    feasible=[p for p in points if p[\'FP_per_image\']<=fp_per_image]\n    best=max(feasible,key=selection_key)\n    return dict(best=best,points=points,fp_budget=fp_per_image,match_iou=iou,\n                small_definition=dict(space=\'원본 이미지\',low_inclusive=small_low,high_exclusive=small_high,measure=\'sqrt(width*height)\'))\n\n\ndef selection_key(point):\n    return point[\'small_swimmer_recall\'],point[\'person_recall\'],-point[\'FP_per_image\']\n', 'train.py': '"""검토 승인된 수영자 자료로 학습하고 작은 수영자 재현율 최적 모델을 보존한다."""\nimport argparse\nimport json\nimport shutil\nfrom pathlib import Path\nfrom recall_selection import digest,verify_bundle,select_operating_point,selection_key\n\n\ndef patch_training_source(repo):\n    replacements=[(\'mmdet/core/bbox/samplers/ic_base_sampler.py\',\n        \'        iod_max_overlaps, _ = iod_overlaps.max(dim=0)\',\n        \'        iod_max_overlaps = iod_overlaps.max(dim=0)[0] if gt_bboxes.numel() else bboxes.new_zeros((bboxes.shape[0],))\'),\n        (\'mmdet/models/module/heatmap.py\',\'threshold = -bg_score_sorted[bg_num - 1]\',\n         \'threshold = -bg_score_sorted[min(bg_num, len(bg_score_sorted)) - 1] if bg_num > 0 and len(bg_score_sorted) else float("inf")\')]\n    for file,old,new in replacements:\n        path=repo/file;text=path.read_text(encoding=\'utf-8\')\n        if old in text:path.write_text(text.replace(old,new),encoding=\'utf-8\')\n        elif new not in text:raise RuntimeError(\'학습 소스가 예상과 다릅니다: \'+file)\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    for name in (\'repo\',\'data\',\'checkpoint\',\'output\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--epochs\',type=int,default=3)\n    p.add_argument(\'--learning-rate\',type=float,default=1e-5)\n    p.add_argument(\'--fp-per-image\',type=float,default=4.)\n    p.add_argument(\'--match-iou\',type=float,default=.3)\n    p.add_argument(\'--patience\',type=int,default=3)\n    p.add_argument(\'--mode\',choices=(\'train\',\'smoke\'),default=\'train\')\n    p.add_argument(\'--smoke-steps\',type=int,default=2)\n    p.add_argument(\'--resume\',action=\'store_true\')\n    args=p.parse_args()\n    if args.epochs<1 or args.smoke_steps<1 or args.patience<1 or args.learning_rate<=0 or args.fp_per_image<0:\n        raise ValueError(\'학습·검증 설정의 범위를 확인하세요.\')\n    root=Path(args.data).resolve();output=Path(args.output).resolve()\n    manifest,raw=verify_bundle(root)\n    print(\'승인 데이터 확인:\',{s:len(r[\'images\']) for s,r in raw.items()},flush=True)\n    output.mkdir(parents=True,exist_ok=True)\n    source=Path(args.repo).resolve();repo=output/\'training_source\'\n    if not repo.exists():shutil.copytree(source,repo,ignore=shutil.ignore_patterns(\'.git\',\'__pycache__\',\'*.pyc\'))\n    patch_training_source(repo)\n    import sys\n    sys.path.insert(0,str(repo))\n    import torch\n    from mmcv import Config\n    from mmcv.parallel import MMDataParallel\n    from mmcv.runner import Hook,build_runner,build_optimizer\n    from mmdet.apis import set_random_seed,single_gpu_test\n    from mmdet.datasets import build_dataset,build_dataloader\n    from mmdet.models import build_detector\n    from mmdet.utils import get_root_logger\n    if not torch.cuda.is_available():raise RuntimeError(\'GPU 런타임이 필요합니다.\')\n    torch.set_num_threads(2)\n    cfg=Config.fromfile(str(repo/\'configs/sspnet/faster_rcnn_r50_sspnet_1x_coco.py\'))\n    cfg.model.pretrained=None;cfg.model.backbone.frozen_stages=3\n    cfg.test_cfg.rcnn.score_thr=.001;cfg.test_cfg.rcnn.nms.min_score=.001;cfg.test_cfg.rcnn.max_per_img=500\n    normalize=dict(type=\'Normalize\',mean=[123.675,116.28,103.53],std=[58.395,57.12,57.375],to_rgb=True)\n    train_pipeline=[dict(type=\'LoadImageFromFile\'),dict(type=\'LoadAnnotations\',with_bbox=True),\n        dict(type=\'Resize\',img_scale=[(640,512),(960,768),(1280,1024)],multiscale_mode=\'value\',keep_ratio=True),\n        dict(type=\'RandomFlip\',flip_ratio=.5),normalize,dict(type=\'Pad\',size_divisor=32),dict(type=\'DefaultFormatBundle\'),\n        dict(type=\'Collect\',keys=[\'img\',\'gt_bboxes\',\'gt_labels\',\'gt_bboxes_ignore\'])]\n    val_pipeline=[dict(type=\'LoadImageFromFile\'),dict(type=\'MultiScaleFlipAug\',img_scale=(1280,1024),flip=False,\n        transforms=[dict(type=\'Resize\',keep_ratio=True),dict(type=\'RandomFlip\'),normalize,dict(type=\'Pad\',size_divisor=32),\n                    dict(type=\'ImageToTensor\',keys=[\'img\']),dict(type=\'Collect\',keys=[\'img\'])])]\n    dataset_cfg=lambda split,pipeline:dict(type=\'CocoDataset\',ann_file=str(root/(split+\'.json\')),img_prefix=str(root)+\'/\',\n                                         classes=(\'person\',),filter_empty_gt=False,pipeline=pipeline)\n    cfg.data=dict(samples_per_gpu=1,workers_per_gpu=0,train=dataset_cfg(\'train\',train_pipeline),val=dataset_cfg(\'val\',val_pipeline))\n    cfg.optimizer=dict(type=\'SGD\',lr=args.learning_rate,momentum=.9,weight_decay=.0001,\n                       paramwise_cfg=dict(custom_keys={\'backbone\':dict(lr_mult=.1),\'neck\':dict(lr_mult=.1)}))\n    cfg.optimizer_config=dict(grad_clip=dict(max_norm=35,norm_type=2));cfg.lr_config=dict(policy=\'fixed\')\n    cfg.workflow=[(\'train\',1)];cfg.gpu_ids=[0];cfg.seed=0;cfg.work_dir=str(output)\n    cfg.runner=dict(type=\'EpochBasedRunner\',max_epochs=1 if args.mode==\'smoke\' else args.epochs)\n    cfg.pop(\'total_epochs\',None)\n    cfg.checkpoint_config=dict(interval=1,by_epoch=True,max_keep_ckpts=2,create_symlink=False)\n    cfg.log_config=dict(interval=1 if args.mode==\'smoke\' else 10,hooks=[dict(type=\'TextLoggerHook\')])\n    signature=dict(quality_manifest_sha256=digest(root/\'quality_manifest.json\'),initial_sha256=digest(args.checkpoint),\n        train_code_sha256=digest(__file__),selection_code_sha256=digest(Path(__file__).with_name(\'recall_selection.py\')),\n        mode=args.mode,epochs=args.epochs,learning_rate=args.learning_rate,fp_per_image=args.fp_per_image,match_iou=args.match_iou,\n        patience=args.patience,smoke_steps=args.smoke_steps,frozen_stages=3,seed=0)\n    signature_path=output/\'experiment.json\'\n    if args.resume:\n        if not signature_path.exists() or json.loads(signature_path.read_text())!=signature or not (output/\'latest.pth\').exists():\n            raise ValueError(\'재개할 실험의 설정·최신 체크포인트를 확인하세요.\')\n    elif signature_path.exists():raise ValueError(\'기존 실행 폴더입니다. --resume 또는 새 출력 폴더를 사용하세요.\')\n    signature_path.write_text(json.dumps(signature,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    cfg.dump(str(output/\'resolved_config.py\'))\n    set_random_seed(0,deterministic=True)\n    trainset=build_dataset(cfg.data.train);valset=build_dataset(cfg.data.val,dict(test_mode=True))\n    if args.mode==\'smoke\':\n        trainset.data_infos=trainset.data_infos[:args.smoke_steps];trainset.img_ids=trainset.img_ids[:args.smoke_steps]\n        trainset.flag=trainset.flag[:args.smoke_steps]\n    trainloader=build_dataloader(trainset,1,0,num_gpus=1,dist=False,seed=0)\n    valloader=build_dataloader(valset,1,0,num_gpus=1,dist=False,shuffle=False)\n    model=build_detector(cfg.model,train_cfg=cfg.train_cfg,test_cfg=cfg.test_cfg)\n    checkpoint=torch.load(args.checkpoint,map_location=\'cpu\');state=checkpoint.get(\'state_dict\',checkpoint)\n    if all(k.startswith(\'module.\') for k in state):state={k[7:]:v for k,v in state.items()}\n    model.load_state_dict(state,strict=True);model.CLASSES=(\'person\',)\n    model=MMDataParallel(model.cuda(),device_ids=[0])\n    runner=build_runner(cfg.runner,default_args=dict(model=model,optimizer=build_optimizer(model,cfg.optimizer),\n        work_dir=str(output),logger=get_root_logger(log_level=\'INFO\'),meta=dict(config=cfg.pretty_text,CLASSES=(\'person\',),quality_manifest_sha256=signature[\'quality_manifest_sha256\'])))\n    runner.register_training_hooks(cfg.lr_config,cfg.optimizer_config,cfg.checkpoint_config,cfg.log_config)\n\n    class SmallSwimmerValidation(Hook):\n        """원본과 매 epoch를 실제 작은 수영자 재현율로 검증한다."""\n        def __init__(self):\n            self.best=None;self.stale=0;self.history=[]\n            if args.resume:\n                self.best=json.loads((output/\'best_selection.json\').read_text())\n                self.history=json.loads((output/\'validation_history.json\').read_text())\n                self.stale=self.best.get(\'stale_epochs\',0)\n\n        def before_run(self,runner):self.evaluate(runner,runner.epoch)\n\n        def after_train_epoch(self,runner):\n            self.evaluate(runner,runner.epoch+1)\n            # Windows 바인드 마운트에서도 재개 파일을 사용할 수 있도록 심볼릭 링크 대신 저장한다.\n            runner.save_checkpoint(str(output),filename_tmpl=\'latest.pth\',create_symlink=False)\n            if self.stale>=args.patience:\n                print(\'검증 재현율 개선이 없어 학습을 조기 종료합니다.\',flush=True)\n                runner._max_epochs=runner.epoch+1\n\n        def evaluate(self,runner,epoch):\n            print(f\'작은 수영자 검증 시작: epoch {epoch}\',flush=True)\n            results=single_gpu_test(runner.model,valloader)\n            predictions=[]\n            for im,result in zip(valset.data_infos,results):\n                if len(result)!=1:raise ValueError(\'단일 사람 클래스 출력이 아닙니다.\')\n                for x,y,r,b,score in result[0]:\n                    if r>x and b>y:\n                        predictions.append(dict(image_id=im[\'id\'],category_id=1,bbox=[float(x),float(y),float(r-x),float(b-y)],score=float(score)))\n            result=select_operating_point(raw[\'val\'],predictions,args.fp_per_image,args.match_iou)\n            point=result[\'best\'];improved=self.best is None or selection_key(point)>selection_key(self.best[\'point\'])\n            if improved:\n                self.stale=0\n                self.best=dict(epoch=epoch,point=point,definition=result[\'small_definition\'],fp_budget=args.fp_per_image,\n                               match_iou=args.match_iou,quality_manifest_sha256=signature[\'quality_manifest_sha256\'])\n                runner.save_checkpoint(str(output),filename_tmpl=\'best_small_swimmer.pth\',create_symlink=False,\n                                       meta=dict(recall_selection=self.best))\n            elif not self.history or self.history[-1][\'epoch\']!=epoch:self.stale+=1\n            self.best[\'stale_epochs\']=self.stale\n            row=dict(epoch=epoch,selected=point,improved=improved,threshold_sweep=result[\'points\'])\n            self.history=[v for v in self.history if v[\'epoch\']!=epoch]+[row]\n            for name,value in [(\'best_selection.json\',self.best),(\'validation_history.json\',self.history)]:\n                tmp=output/(name+\'.tmp\');tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding=\'utf-8\');tmp.replace(output/name)\n            print(f\'epoch {epoch}: 작은 수영자 {point["small_swimmer_hits"]}/{point["small_swimmer_positives"]} \'\n                  f\'재현율 {point["small_swimmer_recall"]:.1%}, 전체 {point["person_recall"]:.1%}, \'\n                  f\'FP/이미지 {point["FP_per_image"]:.2f}, 임계값 {point["threshold"]:.4f}, 최적 갱신 {improved}\',flush=True)\n            runner.model.train()\n\n    runner.register_hook(SmallSwimmerValidation(),priority=\'LOW\')\n    if args.resume:runner.resume(str(output/\'latest.pth\'))\n    runner.run([trainloader],cfg.workflow)\n    best=json.loads((output/\'best_selection.json\').read_text())\n    print(\'선택한 최적 epoch:\',best[\'epoch\'],\'가중치:\',output/\'best_small_swimmer.pth\',flush=True)\n    if best[\'epoch\']==0:print(\'추가 학습이 원본 재현율을 개선하지 못해 원본 모델을 유지했습니다.\',flush=True)\n\n\nif __name__==\'__main__\':main()\n'}
HELPERS=WORK/'helpers';HELPERS.mkdir(exist_ok=True)
for name,source in HELPER_SOURCES.items():(HELPERS/name).write_text(source,encoding='utf-8')
subprocess.run([sys.executable,str(HELPERS/'setup_environment.py')],check=True)
PYTHON=WORK/'venv/bin/python';REPO=WORK/'SSPNet'
ENV=os.environ.copy();ENV.pop('PYTHONHOME',None)
ENV.update(PYTHONNOUSERSITE='1',PYTHONPATH=str(HELPERS),MPLBACKEND='Agg',PYTHONUNBUFFERED='1')
def checked(args):subprocess.run([str(v) for v in args],env=ENV,check=True)
CHECKPOINT=STORE/'weights/faster_rcnn_sspnet.pth';CHECKPOINT.parent.mkdir(exist_ok=True)
if not CHECKPOINT.exists():
    checked([PYTHON,'-c',"import gdown; assert gdown.download(id='1IfPCt5xZqqBJ3sYVIuD5F9l29Jcy2Hn1',output=%r,quiet=False)"%str(CHECKPOINT)])
import hashlib
if hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()!='912cc5b6f8a45b0f4521c9bf325868ed10dd4757497fb6ba8843fdbd26b09e4a':
    raise ValueError('원본 가중치 해시가 다릅니다.')


In [ ]:
DATA_ZIP=Path(DATA_ZIP_PATH) if DATA_ZIP_PATH else WORK/'recall_v1.zip'
if not DATA_ZIP.is_file():
    print('검토 승인된 recall_v1.zip 한 개를 선택하세요.')
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError('ZIP 한 개가 필요합니다.')
    DATA_ZIP.write_bytes(next(iter(uploaded.values())))
DATA=WORK/'recall_v1';DATA.mkdir(exist_ok=True)
with zipfile.ZipFile(DATA_ZIP) as archive:
    for item in archive.infolist():
        target=(DATA/item.filename).resolve()
        if target!=DATA.resolve() and DATA.resolve() not in target.parents:raise ValueError('ZIP 경로가 잘못됐습니다.')
    archive.extractall(DATA)
checked([PYTHON,'-c',"from recall_selection import verify_bundle; verify_bundle(%r); print('검토 승인·해시·촬영 그룹 분리 확인 완료')"%str(DATA)])


시험 학습은 실제 2회 업데이트와 원본·학습 후 검증을 실행합니다. 본 학습은 원본에서 별도로 시작합니다.
재개할 때는 같은 실행 이름·설정으로 환경과 데이터를 준비하고 시험 셀을 건너뛴 뒤 본 학습 셀의 RESUME을 켜세요.


In [ ]:
COMMON=[PYTHON,HELPERS/'train.py','--repo',REPO,'--data',DATA,'--checkpoint',CHECKPOINT,
        '--epochs',EPOCHS,'--learning-rate',LEARNING_RATE,'--fp-per-image',FP_PER_IMAGE]
checked(COMMON+['--output',STORE/'training'/RUN_NAME/'smoke','--mode','smoke','--smoke-steps',2])


In [ ]:
RESUME=False
OUTPUT=STORE/'training'/RUN_NAME/'main'
args=COMMON+['--output',OUTPUT]
if RESUME:args+=['--resume']
checked(args)
result=json.loads((OUTPUT/'best_selection.json').read_text())
print('최적 가중치:',OUTPUT/'best_small_swimmer.pth')
print('최적 epoch·임계값:',result['epoch'],result['point']['threshold'])
print('원본 유지 여부:',result['epoch']==0)
